In [1]:
# notebook designed to test the cache.py script when it was made only for muons
# does it succesfully save the tth MVA scores on eos?
import sys
from pathlib import Path

import awkward as ak
import numpy as np
import pandas as pd

sys.path.insert(
    0,
    str(Path.cwd() / "analysis" / "tthMVA")
)

import cache

Beware that when running from coffea-casa, you have to manually put in the target directory's 'userpath' for writing the tth MVA scores in analysis/tthMVA/cache.py
Your current path to write the tthMVA score cache to is '/home/cms-jovyan/ua-cms-higgscharm/analysis/data/tthMVA_2022-2023_retrained/tthMVA_cache'


In [2]:
# see whether the cache recalculates any MVA score
class DummyEvaluator:

    def __init__(self):

        self.calls = 0

    def evaluate_array(self, values):

        self.calls += 1

        values = np.asarray(values)

        return np.arange(
            len(values),
            dtype=np.float32,
        )

In [3]:
# make fake events
events = ak.Array(
    {
        "run": [1, 1],

        "luminosityBlock": [100, 100],

        "event": [1000, 1001],

        "Muon": [
            [
                {
                    "pt": 25.,
                    "eta": 0.1,
                    "pfRelIso03_all": 0.05,
                    "miniPFRelIso_chg": 0.01,
                    "miniPFRelIso_all": 0.02,
                    "jetNDauCharged": 5,
                    "jetPtRelv2": 10.,
                    "jetIdx": 0,
                    "jetRelIso": 0.2,
                    "sip3d": 1.0,
                    "dxy": 0.001,
                    "dz": 0.01,
                    "segmentComp": 0.9,
                },

                {
                    "pt": 30.,
                    "eta": -0.2,
                    "pfRelIso03_all": 0.03,
                    "miniPFRelIso_chg": 0.01,
                    "miniPFRelIso_all": 0.015,
                    "jetNDauCharged": 4,
                    "jetPtRelv2": 20.,
                    "jetIdx": 1,
                    "jetRelIso": 0.1,
                    "sip3d": 0.5,
                    "dxy": 0.002,
                    "dz": 0.02,
                    "segmentComp": 0.95,
                },
            ],

            [
                {
                    "pt": 50.,
                    "eta": 0.5,
                    "pfRelIso03_all": 0.01,
                    "miniPFRelIso_chg": 0.005,
                    "miniPFRelIso_all": 0.008,
                    "jetNDauCharged": 8,
                    "jetPtRelv2": 15.,
                    "jetIdx": 0,
                    "jetRelIso": 0.3,
                    "sip3d": 0.2,
                    "dxy": 0.001,
                    "dz": 0.005,
                    "segmentComp": 0.99,
                }
            ]
        ],

        "Jet": [
            [
                {"btagDeepFlavB": 0.2},
                {"btagDeepFlavB": 0.8},
            ],

            [
                {"btagDeepFlavB": 0.4},
            ]
        ]
    }
)

In [4]:
# create cache object and remove old cache 
cache_obj = cache.TTHMVACache(
    "/home/cms-jovyan/ua-cms-higgscharm/analysis/data/tthMVA_2022-2023_retrained/Muon-mvaTTH.2022EE.weights.json"
)

cache_obj.evaluator = DummyEvaluator()

# you need to manually remove old cache, as coffea-casa has no direct access to eos

In [5]:
# compute scores
scores1 = cache_obj.get_scores(
    events,
    dataset="TESTDATASET",
    filename="TESTFILE.root",
)

print(scores1)

print(
    "Evaluator calls:",
    cache_obj.evaluator.calls
)
# check in the cache_base that the proper directory and files are created


CACHED KEYS:

EVENT KEYS:
(1, 100, 1000, 0) MISSING
(1, 100, 1000, 1) MISSING
(1, 100, 1001, 0) MISSING
Missing muons: 3
[[0, 1], [2]]
Evaluator calls: 1


/home/cms-jovyan/ua-cms-higgscharm/analysis/tthMVA/cache.py:333: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  cache = pd.concat(


In [6]:
# take the full path of the created file and paste it below in cache_file
cache_file = "/home/cms-jovyan/ua-cms-higgscharm/analysis/data/tthMVA_2022-2023_retrained/tthMVA_cache/Muon-mvaTTH.2022EE.weights/3c5552f67796dac9042c8a50a86e4324c5f4640a.parquet"
pd.read_parquet(cache_file)
# output is as expected

,run,lumi,event,muon,tthMVA
0,1,100,1000,0,0.0
1,1,100,1000,1,1.0
2,1,100,1001,0,2.0


In [7]:
# now test if the tth MVA scores are accessible and called from the cache when calling again
scores2 = cache_obj.get_scores(
    events,
    dataset="TESTDATASET",
    filename="TESTFILE.root",
)

print(scores2)

print(
    "Evaluator calls:",
    cache_obj.evaluator.calls
)
# evaluator calls is still the saem, so no additional, evaluation was done -> success!


CACHED KEYS:
(1, 100, 1000, 0)
(1, 100, 1000, 1)
(1, 100, 1001, 0)

EVENT KEYS:
(1, 100, 1000, 0) CACHED
(1, 100, 1000, 1) CACHED
(1, 100, 1001, 0) CACHED
Missing muons: 0
[[0, 1], [2]]
Evaluator calls: 1


In [8]:
# are scores the same as first evaluation?
ak.all(scores1 == scores2)
# Yes!

True

In [27]:
# add additional muon and check that only that one is run thropiugh the machinery to calculate the tth MVA score
events2 = ak.Array(
    {
        "run": [1, 1, 1],
        "luminosityBlock": [100, 100, 100],
        "event": [1000, 1001, 1002],

        "Muon": [
            events.Muon[0],
            events.Muon[1],

            [
                {
                    "pt": 60.,
                    "eta": 0.2,
                    "pfRelIso03_all": 0.01,
                    "miniPFRelIso_chg": 0.005,
                    "miniPFRelIso_all": 0.006,
                    "jetNDauCharged": 5,
                    "jetPtRelv2": 12.,
                    "jetIdx": 0,
                    "jetRelIso": 0.1,
                    "sip3d": 0.3,
                    "dxy": 0.001,
                    "dz": 0.002,
                    "segmentComp": 0.95,
                }
            ]
        ],

        "Jet": [
            events.Jet[0],
            events.Jet[1],
            [{"btagDeepFlavB": 0.3}],
        ],
    }
)

In [10]:
scores3 = cache_obj.get_scores(
    events2,
    dataset="TESTDATASET",
    filename="TESTFILE.root",
)

print(
    "Evaluator calls:",
    cache_obj.evaluator.calls
)
dataset="TESTDATASET"
filename="TESTFILE.root"
cache_file = cache_obj._cache_file(dataset, filename)

print("Cache file:")
print(cache_file)
print("Exists:", cache_file.exists())

cache_df = pd.read_parquet(cache_file)

print("\nNumber of rows:", len(cache_df))
print(cache_df[["run", "lumi", "event", "muon", "tthMVA"]])
# now evaluator needs to be called again and evaluator calls goes 1 up


CACHED KEYS:
(1, 100, 1000, 0)
(1, 100, 1000, 1)
(1, 100, 1001, 0)

EVENT KEYS:
(1, 100, 1000, 0) CACHED
(1, 100, 1000, 1) CACHED
(1, 100, 1001, 0) CACHED
(1, 100, 1002, 0) MISSING
Missing muons: 1
Evaluator calls: 2
Cache file:
/home/cms-jovyan/ua-cms-higgscharm/analysis/data/tthMVA_2022-2023_retrained/tthMVA_cache/Muon-mvaTTH.2022EE.weights/3c5552f67796dac9042c8a50a86e4324c5f4640a.parquet
Exists: True

Number of rows: 4
   run  lumi  event  muon  tthMVA
0    1   100   1000     0     0.0
1    1   100   1000     1     1.0
2    1   100   1001     0     2.0
3    1   100   1002     0     0.0


In [11]:
# check again what the output is 
cache_file = "/home/cms-jovyan/ua-cms-higgscharm/analysis/data/tthMVA_2022-2023_retrained/tthMVA_cache/Muon-mvaTTH.2022EE.weights/3c5552f67796dac9042c8a50a86e4324c5f4640a.parquet"
pd.read_parquet(cache_file)

,run,lumi,event,muon,tthMVA
0,1,100,1000,0,0.0
1,1,100,1000,1,1.0
2,1,100,1001,0,2.0
3,1,100,1002,0,0.0


In [12]:
cache_df = pd.read_parquet(cache_file)
print(len(cache_df))

4


In [13]:
# see what difference is between muons in each events fake dataset
print("Original:")
print(ak.num(events.Muon))

print("Modified:")
print(ak.num(events2.Muon))

Original:
[2, 1]
Modified:
[2, 1, 1]


In [14]:
# run to delete cached file:
"""
dataset="TESTDATASET"
filename="TESTFILE.root"
cache_file = cache_obj._cache_file(dataset, filename)

print(cache_file)
if cache_file.exists():
    cache_file.unlink()
    print("Deleted existing cache.")
else:
    print("Cache did not exist.")
"""

'\ndataset="TESTDATASET"\nfilename="TESTFILE.root"\ncache_file = cache_obj._cache_file(dataset, filename)\n\nprint(cache_file)\nif cache_file.exists():\n    cache_file.unlink()\n    print("Deleted existing cache.")\nelse:\n    print("Cache did not exist.")\n'

In [15]:
cache_file = "/home/cms-jovyan/ua-cms-higgscharm/analysis/data/tthMVA_2022-2023_retrained/tthMVA_cache/Muon-mvaTTH.2022EE.weights/3c5552f67796dac9042c8a50a86e4324c5f4640a.parquet"
pd.read_parquet(cache_file)

,run,lumi,event,muon,tthMVA
0,1,100,1000,0,0.0
1,1,100,1000,1,1.0
2,1,100,1001,0,2.0
3,1,100,1002,0,0.0


In [32]:
# test the code now after cache.py was updated to also do this for electrons. 
# Now the file is saved in ua-cms-higgscharm/analysis/data/tthMVA_2022-2023_retrained/tthMVA_cache/e9ffddbd9e29aa1e3a6e1404c903d760962cf934.parquet
import sys
from pathlib import Path

import awkward as ak
import numpy as np
import pandas as pd

sys.path.insert(
    0,
    str(Path.cwd() / "analysis" / "tthMVA")
)

import cache

In [33]:
# see whether the cache recalculates any MVA score
class DummyEvaluator:

    def __init__(self):

        self.calls = 0

    def evaluate_array(self, values):

        self.calls += 1

        values = np.asarray(values)

        return np.arange(
            len(values),
            dtype=np.float32,
        )

In [34]:
# make fake events
events = ak.Array(
    {
        "run": [1, 1],

        "luminosityBlock": [100, 100],

        "event": [1000, 1001],

        # --------------------------------------------------------
        # Muons
        # --------------------------------------------------------

        "Muon": [
            [
                {
                    "pt": 25.0,
                    "eta": 0.1,
                    "pfRelIso03_all": 0.05,
                    "miniPFRelIso_chg": 0.01,
                    "miniPFRelIso_all": 0.02,
                    "jetNDauCharged": 5,
                    "jetPtRelv2": 10.0,
                    "jetIdx": 0,
                    "jetRelIso": 0.2,
                    "sip3d": 1.0,
                    "dxy": 0.001,
                    "dz": 0.01,
                    "segmentComp": 0.9,
                },

                {
                    "pt": 30.0,
                    "eta": -0.2,
                    "pfRelIso03_all": 0.03,
                    "miniPFRelIso_chg": 0.01,
                    "miniPFRelIso_all": 0.015,
                    "jetNDauCharged": 4,
                    "jetPtRelv2": 20.0,
                    "jetIdx": 1,
                    "jetRelIso": 0.1,
                    "sip3d": 0.5,
                    "dxy": 0.002,
                    "dz": 0.02,
                    "segmentComp": 0.95,
                },
            ],

            [
                {
                    "pt": 50.0,
                    "eta": 0.5,
                    "pfRelIso03_all": 0.01,
                    "miniPFRelIso_chg": 0.005,
                    "miniPFRelIso_all": 0.008,
                    "jetNDauCharged": 8,
                    "jetPtRelv2": 15.0,
                    "jetIdx": 0,
                    "jetRelIso": 0.3,
                    "sip3d": 0.2,
                    "dxy": 0.001,
                    "dz": 0.005,
                    "segmentComp": 0.99,
                }
            ]
        ],

        # --------------------------------------------------------
        # Electrons
        # --------------------------------------------------------

        "Electron": [
            [
                {
                    "pt": 22.0,
                    "eta": -0.3,
                    "pfRelIso03_all": 0.04,
                    "miniPFRelIso_chg": 0.012,
                    "miniPFRelIso_all": 0.025,
                    "jetNDauCharged": 3,
                    "jetPtRelv2": 8.0,
                    "jetIdx": 0,
                    "jetRelIso": 0.25,
                    "sip3d": 0.8,
                    "dxy": 0.0015,
                    "dz": 0.015,
                    "mvaIso": 0.85,
                },
            ],

            [
                {
                    "pt": 35.0,
                    "eta": 0.4,
                    "pfRelIso03_all": 0.02,
                    "miniPFRelIso_chg": 0.008,
                    "miniPFRelIso_all": 0.012,
                    "jetNDauCharged": 6,
                    "jetPtRelv2": 12.0,
                    "jetIdx": 0,
                    "jetRelIso": 0.15,
                    "sip3d": 0.4,
                    "dxy": 0.002,
                    "dz": 0.008,
                    "mvaIso": 0.92,
                },

                {
                    "pt": 45.0,
                    "eta": -0.6,
                    "pfRelIso03_all": 0.015,
                    "miniPFRelIso_chg": 0.005,
                    "miniPFRelIso_all": 0.009,
                    "jetNDauCharged": 7,
                    "jetPtRelv2": 18.0,
                    "jetIdx": -1,
                    "jetRelIso": 0.05,
                    "sip3d": 0.3,
                    "dxy": 0.001,
                    "dz": 0.006,
                    "mvaIso": 0.97,
                },
            ]
        ],

        # --------------------------------------------------------
        # Jets
        # --------------------------------------------------------

        "Jet": [
            [
                {"btagDeepFlavB": 0.2},
                {"btagDeepFlavB": 0.8},
            ],

            [
                {"btagDeepFlavB": 0.4},
            ],
        ],
    }
)

In [35]:
# create cache object and remove old cache 
cache_obj_muons = cache.TTHMVACache(
    "/home/cms-jovyan/ua-cms-higgscharm/analysis/data/tthMVA_2022-2023_retrained/Muon-mvaTTH.2022EE.weights.json"
)
cache_obj_electrons = cache.TTHMVACache(
    "/home/cms-jovyan/ua-cms-higgscharm/analysis/data/tthMVA_2022-2023_retrained/Electron-mvaTTH.2022EE.weights_mvaISO.json"
)

cache_obj_muons.evaluator = DummyEvaluator()
cache_obj_electrons.evaluator = DummyEvaluator()

# you need to manually remove old cache, as coffea-casa has no direct access to eos

In [36]:
# compute scores
scores1 = cache_obj_muons.get_scores(
    events,
    dataset="TESTDATASET",
    filename="TESTFILE.root",
    particle="muons",
)

print(scores1)

print(
    "Evaluator calls:",
    cache_obj_muons.evaluator.calls
)

scores1e = cache_obj_electrons.get_scores(
    events,
    dataset="TESTDATASET",
    filename="TESTFILE.root",
    particle="electrons",
)

print(scores1e)

print(
    "Evaluator calls:",
    cache_obj_electrons.evaluator.calls
)
# check in the cache_base that the proper directory and files are created

[[0, 1], [2]]
Evaluator calls: 1
[[0], [1, 2]]
Evaluator calls: 1


/home/cms-jovyan/ua-cms-higgscharm/analysis/tthMVA/cache.py:450: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  cache = pd.concat(
/home/cms-jovyan/ua-cms-higgscharm/analysis/tthMVA/cache.py:450: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  cache = pd.concat(


In [37]:
# take the full path of the created file and paste it below in cache_file
cache_file = "/home/cms-jovyan/ua-cms-higgscharm/analysis/data/tthMVA_2022-2023_retrained/tthMVA_cache/3c5552f67796dac9042c8a50a86e4324c5f4640a.parquet"
pd.read_parquet(cache_file)
# output is as expected

,run,lumi,event,muon,muon_tthMVA,electron,electron_tthMVA
0,1,100,1000,0.0,0.0,NaN,NaN
1,1,100,1000,1.0,1.0,NaN,NaN
2,1,100,1001,0.0,2.0,NaN,NaN
3,1,100,1000,NaN,NaN,0.0,0.0
4,1,100,1001,NaN,NaN,0.0,1.0
5,1,100,1001,NaN,NaN,1.0,2.0


In [38]:
# now test if the tth MVA scores are accessible and called from the cache when calling again
scores2 = cache_obj_muons.get_scores(
    events,
    dataset="TESTDATASET",
    filename="TESTFILE.root",
    particle="muons",
)

print(scores2)

print(
    "Evaluator calls:",
    cache_obj_muons.evaluator.calls
)

scores2e = cache_obj_electrons.get_scores(
    events,
    dataset="TESTDATASET",
    filename="TESTFILE.root",
    particle="electrons",
)

print(scores2e)

print(
    "Evaluator calls:",
    cache_obj_electrons.evaluator.calls
)
# evaluator calls is still the same, so no additional, evaluation was done -> success!

[[0, 1], [2]]
Evaluator calls: 1
[[0], [1, 2]]
Evaluator calls: 1


In [39]:
# are scores the same as first evaluation?
ak.all(scores1 == scores2)
# Yes!

True

In [40]:
# Add an additional event containing both a new muon and a new electron.
events2 = ak.Array(
    {
        "run": [1, 1, 1],
        "luminosityBlock": [100, 100, 100],
        "event": [1000, 1001, 1002],

        "Muon": [
            # Event 1000: same muons as in events
            events.Muon[0],

            # Event 1001: same muon as in events
            events.Muon[1],

            # Event 1002: NEW muon
            [
                {
                    "pt": 60.,
                    "eta": 0.2,
                    "pfRelIso03_all": 0.01,
                    "miniPFRelIso_chg": 0.005,
                    "miniPFRelIso_all": 0.006,
                    "jetNDauCharged": 5,
                    "jetPtRelv2": 12.,
                    "jetIdx": 0,
                    "jetRelIso": 0.1,
                    "sip3d": 0.3,
                    "dxy": 0.001,
                    "dz": 0.002,
                    "segmentComp": 0.95,
                }
            ],
        ],

        "Electron": [
            # Event 1000: same electrons as in events
            events.Electron[0],

            # Event 1001: same electrons as in events
            events.Electron[1],

            # Event 1002: NEW electron
            [
                {
                    "pt": 45.,
                    "eta": -0.3,
                    "pfRelIso03_all": 0.02,
                    "miniPFRelIso_chg": 0.008,
                    "miniPFRelIso_all": 0.012,
                    "jetNDauCharged": 6,
                    "jetPtRelv2": 14.,
                    "jetIdx": 0,
                    "jetRelIso": 0.15,
                    "sip3d": 0.4,
                    "dxy": 0.002,
                    "dz": 0.004,
                    "mvaIso": 0.85,
                }
            ],
        ],

        "Jet": [
            # Event 1000
            events.Jet[0],

            # Event 1001
            events.Jet[1],

            # Event 1002: jet associated with both new objects
            [
                {"btagDeepFlavB": 0.3},
            ],
        ],
    }
)

In [41]:
# muons
scores3_m = cache_obj_muons.get_scores(
    events2,
    dataset="TESTDATASET",
    filename="TESTFILE.root",
    particle="muons",
)

print(
    "Evaluator calls:",
    cache_obj_muons.evaluator.calls
)
dataset="TESTDATASET"
filename="TESTFILE.root"
cache_file_m = cache_obj_muons._cache_file(dataset, filename)

print("Cache file:")
print(cache_file)
print("Exists:", cache_file_m.exists())

cache_df_m = pd.read_parquet(cache_file_m)

print("\nNumber of rows:", len(cache_df_m))
print(cache_df_m[["run", "lumi", "event", "muon", "muon_tthMVA"]])

# electrons
scores3_e = cache_obj_electrons.get_scores(
    events2,
    dataset="TESTDATASET",
    filename="TESTFILE.root",
    particle="electrons",
)

print(
    "Evaluator calls:",
    cache_obj_electrons.evaluator.calls
)
dataset="TESTDATASET"
filename="TESTFILE.root"
cache_file_e = cache_obj_electrons._cache_file(dataset, filename)

print("Cache file:")
print(cache_file)
print("Exists:", cache_file_e.exists())

cache_df_e = pd.read_parquet(cache_file_e)

print("\nNumber of rows:", len(cache_df_e))
print(cache_df_e[["run", "lumi", "event", "electron", "electron_tthMVA"]])
# now evaluator needs to be called again and evaluator calls goes 1 up

Evaluator calls: 2
Cache file:
/home/cms-jovyan/ua-cms-higgscharm/analysis/data/tthMVA_2022-2023_retrained/tthMVA_cache/3c5552f67796dac9042c8a50a86e4324c5f4640a.parquet
Exists: True

Number of rows: 7
   run  lumi  event  muon  muon_tthMVA
0    1   100   1000   0.0          0.0
1    1   100   1000   1.0          1.0
2    1   100   1001   0.0          2.0
3    1   100   1000   NaN          NaN
4    1   100   1001   NaN          NaN
5    1   100   1001   NaN          NaN
6    1   100   1002   0.0          0.0
Evaluator calls: 2
Cache file:
/home/cms-jovyan/ua-cms-higgscharm/analysis/data/tthMVA_2022-2023_retrained/tthMVA_cache/3c5552f67796dac9042c8a50a86e4324c5f4640a.parquet
Exists: True

Number of rows: 8
   run  lumi  event  electron  electron_tthMVA
0    1   100   1000       NaN              NaN
1    1   100   1000       NaN              NaN
2    1   100   1001       NaN              NaN
3    1   100   1000       0.0              0.0
4    1   100   1001       0.0              1.0
5   

/home/cms-jovyan/ua-cms-higgscharm/analysis/tthMVA/cache.py:450: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  cache = pd.concat(
/home/cms-jovyan/ua-cms-higgscharm/analysis/tthMVA/cache.py:450: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  cache = pd.concat(


In [42]:
# check again what the output is 
cache_file = "/home/cms-jovyan/ua-cms-higgscharm/analysis/data/tthMVA_2022-2023_retrained/tthMVA_cache/3c5552f67796dac9042c8a50a86e4324c5f4640a.parquet"
pd.read_parquet(cache_file)

,run,lumi,event,muon,muon_tthMVA,electron,electron_tthMVA
0,1,100,1000,0.0,0.0,NaN,NaN
1,1,100,1000,1.0,1.0,NaN,NaN
2,1,100,1001,0.0,2.0,NaN,NaN
3,1,100,1000,NaN,NaN,0.0,0.0
4,1,100,1001,NaN,NaN,0.0,1.0
5,1,100,1001,NaN,NaN,1.0,2.0
6,1,100,1002,0.0,0.0,NaN,NaN
7,1,100,1002,NaN,NaN,0.0,0.0


In [18]:
# run to delete cached file for muons and electrons:
"""
dataset="TESTDATASET"
filename="TESTFILE.root"
cache_file_m = cache_obj_muons._cache_file(dataset, filename)
cache_file_e = cache_obj_electrons._cache_file(dataset, filename)

print(cache_file_m)
if cache_file_m.exists():
    cache_file_m.unlink()
    print("Deleted existing cache.")
else:
    print("Cache did not exist.")

print(cache_file_e)
if cache_file_e.exists():
    cache_file_e.unlink()
    print("Deleted existing cache.")
else:
    print("Cache did not exist.")
"""

/home/cms-jovyan/ua-cms-higgscharm/analysis/data/tthMVA_2022-2023_retrained/tthMVA_cache/3c5552f67796dac9042c8a50a86e4324c5f4640a.parquet
Cache did not exist.
/home/cms-jovyan/ua-cms-higgscharm/analysis/data/tthMVA_2022-2023_retrained/tthMVA_cache/3c5552f67796dac9042c8a50a86e4324c5f4640a.parquet
Cache did not exist.
